In [1]:
!pip install ultralytics opencv-python numpy

  Using cached typing_extensions-4.16.0-py3-none-any.whl.metadata (3.3 kB)
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ----------------------------- ---------- 1.0/1.4 MB 5.1 MB/s eta 0:00:01
   ---------------------------------------- 1.4/1.4 MB 4.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/911.5 kB ? eta -:--:--
   ---------------------------------- ----- 786.4/911.5 kB 4.5 MB/s eta 0:00:01
   ---------------------------------------- 911.5/911.5 kB 3.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/53.9 MB ? eta -:--:--
    --------------------------------------- 0.8/53.9 MB 4.7 MB/s eta 0:00:12
   - -------------------------------------- 1.6/53.9 MB 4.1 MB/s eta 0:00:13
   - -------------------------------------- 2.4/53.9 MB 4.1 MB/s eta 0:00:13
   -- ------------------------------------- 3.1/53.9 MB 4.0 MB/s eta 0:00:13
   --- ------------------------------------ 4.2/53.9 MB 4.0 MB/s eta 0:00:13
   --- -----------

In [2]:
import cv2
import numpy as np
from ultralytics import YOLO
import os

Creating new Ultralytics Settings v0.0.8 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\Avantika\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


In [3]:
model = YOLO("yolov8n-pose.pt")

print("YOLOv8 Pose Model Loaded Successfully")

YOLOv8 Pose Model Loaded Successfully


In [4]:
video_path = "input_video.mp4"

cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print("Error: Video not found")
else:
    print("Video Loaded Successfully")

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

if fps <= 0:
    fps = 30

print("Width:", width)
print("Height:", height)
print("FPS:", fps)

Video Loaded Successfully
Width: 528
Height: 944
FPS: 30.063842135809633


In [5]:
white_output = "black_points_white_background.mp4"
black_output = "white_points_black_background.mp4"

fourcc = cv2.VideoWriter_fourcc(*'mp4v')

out_white = cv2.VideoWriter(
    white_output, fourcc, fps, (width, height)
)

out_black = cv2.VideoWriter(
    black_output, fourcc, fps, (width, height)
)

while cap.isOpened():

    ret, frame = cap.read()

    if not ret:
        break

    results = model.predict(frame, verbose=False)

    white_bg = np.full((height, width, 3), 255, dtype=np.uint8)
    black_bg = np.zeros((height, width, 3), dtype=np.uint8)

    for result in results:

        if result.keypoints is None:
            continue

        keypoints = result.keypoints.xy.cpu().numpy()

        if result.keypoints.conf is not None:
            confidence = result.keypoints.conf.cpu().numpy()
        else:
            confidence = np.ones(
                (keypoints.shape[0], keypoints.shape[1])
            )

        for person_index, person_points in enumerate(keypoints):

            for point_index, point in enumerate(person_points):

                x, y = int(point[0]), int(point[1])

                if confidence[person_index][point_index] < 0.25:
                    continue

                if 0 <= x < width and 0 <= y < height:

                    cv2.circle(white_bg, (x, y), 5, (0, 0, 0), -1)

                    cv2.circle(black_bg, (x, y), 5, (255, 255, 255), -1)

    out_white.write(white_bg)
    out_black.write(black_bg)

cap.release()
out_white.release()
out_black.release()

print("Pose Estimation Completed Successfully!")

Pose Estimation Completed Successfully!


In [10]:
!pip install imageio-ffmpeg


   ---------------------------------------- 0.0/31.2 MB ? eta -:--:--
   - -------------------------------------- 1.0/31.2 MB 5.2 MB/s eta 0:00:06
   -- ------------------------------------- 1.8/31.2 MB 4.5 MB/s eta 0:00:07
   --- ------------------------------------ 2.6/31.2 MB 4.3 MB/s eta 0:00:07
   ---- ----------------------------------- 3.4/31.2 MB 4.1 MB/s eta 0:00:07
   ----- ---------------------------------- 4.2/31.2 MB 4.2 MB/s eta 0:00:07
   ------ --------------------------------- 4.7/31.2 MB 3.9 MB/s eta 0:00:07
   ------- -------------------------------- 5.8/31.2 MB 4.1 MB/s eta 0:00:07
   -------- ------------------------------- 6.6/31.2 MB 4.0 MB/s eta 0:00:07
   --------- ------------------------------ 7.3/31.2 MB 4.0 MB/s eta 0:00:06
   ---------- ----------------------------- 8.1/31.2 MB 4.0 MB/s eta 0:00:06
   ----------- ---------------------------- 8.9/31.2 MB 4.0 MB/s eta 0:00:06
   ------------ --------------------------- 9.7/31.2 MB 4.0 MB/s eta 0:00:06
   ---

In [11]:
import imageio_ffmpeg

ffmpeg_path = imageio_ffmpeg.get_ffmpeg_exe()

import subprocess

subprocess.run([
    ffmpeg_path,
    "-y",
    "-i", "black_points_white_background.mp4",
    "-vcodec", "libx264",
    "-pix_fmt", "yuv420p",
    "fixed_output.mp4"
])

CompletedProcess(args=['C:\\Users\\Avantika\\Downloads\\anaconda\\Lib\\site-packages\\imageio_ffmpeg\\binaries\\ffmpeg-win-x86_64-v7.1.exe', '-y', '-i', 'black_points_white_background.mp4', '-vcodec', 'libx264', '-pix_fmt', 'yuv420p', 'fixed_output.mp4'], returncode=0)

In [8]:
import imageio_ffmpeg
import subprocess
import os

ffmpeg_path = imageio_ffmpeg.get_ffmpeg_exe()

# Convert first output
subprocess.run([
    ffmpeg_path, "-y",
    "-i", "black_points_white_background.mp4",
    "-vcodec", "libx264",
    "-pix_fmt", "yuv420p",
    "final_black_points.mp4"
])

# Convert second output
subprocess.run([
    ffmpeg_path, "-y",
    "-i", "white_points_black_background.mp4",
    "-vcodec", "libx264",
    "-pix_fmt", "yuv420p",
    "final_white_points.mp4"
])

print("Both videos converted successfully!")

Both videos converted successfully!


In [12]:
os.startfile("final_black_points.mp4")
os.startfile("final_white_points.mp4")